# Tool Evalution

multiple agents independently run a single evalution task from an evaluation file

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import re
import time
import traceback
import xml.etree.ElementTree as ET # noqa: S314
from pathlib import Path
from typing import Any

from openai import AzureOpenAI

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

## Prompts

In [4]:
# Embedded evaluator prompt
EVALUATION_PROMPT = """You are an AI assistant with access to tools.

When given a task, you MUST:
1. Use the evailable tools to complete the task
2. Provide summary of each step in your appoarch, wrapped in <summary> tags
3. Provide feeback on the tools provided, wrapped in <feedback> tags
4. Provide your final response, wrapped in <response> tags

Summary Requirements:
- In your <summary> tags, you must explain:
    - The steps you took to complete the task
    - Which tools you used, in what order, and why
    - The inputs you provided to each tool
    - The outputs you received from each tool
    - A summary for how you arrived at the response

Feedback Requirements:
- In your <feedback> tags, provide constructive feedback on the tools:
    - Comment on tool names: Are they clear and descriptive?
    - Comment on input parameters: Are they well-documented? Are required vs optional parameters clear?
    - Comment on descriptions: Do they accurately describe what the tool does?
    - Comment on any errors encounted during tool usage: Did the tool fail to execute? Did the tool return too many tokens?
    - Indentify specific ereas for improvement and explain WHY they would help
    - Be specific and actionable in your suggestions

    Response Requirements:
    - Your response should be concise and directly address what was asked
    - Always wrap your final response in <response> tags
    - If you cannot solve the task return <response>NOT_FOUND</response>
    - For numeric responses, provide just the number
    - For names or text, provide the extract text requested
    - Your response should go last
"""

## Agent Loop

In [5]:
client = AzureOpenAI(
    api_version="2025-04-01-preview",
    azure_endpoint="https://bookingcare-ai-nam-resource.cognitiveservices.azure.com"
)

In [16]:
completion = client.responses.create(
        model="gpt-5.4",
        input= "write a one-sentence bedtime story about a unicorn."
)
print(completion.output[0].content)

[ResponseOutputText(annotations=[], text='A gentle unicorn with a moonlit mane tiptoed through a silver forest, sprinkling sleepy stars over the world until every creature drifted into sweet dreams.', type='output_text', logprobs=[])]


In [23]:
model = "gpt-5.5"

def agent_loop(prompt: str, tools: list[dict[str, Any]] = None) -> tuple[str,dict[str, Any]]:
    """Simplified agent class for tool evaluation"""
    messages = [
        {
            "role": "system",
            "content": EVALUATION_PROMPT
        },
        {
            "role": "user", "content": prompt
        }
    ]

    # Track tool calls with timing
    tool_metrics = {} # {tool_name: {"count": N, "duration": [X1, X2, ...]}}

    def _prepara_tool_result(tool_call_id, tool_result):
        return {
            "type": "function_call_output",
            "call_id": tool_call_id,
            "output": tool_result_str
        }

    while True:
        response = client.responses.create(
            model=model,
            input=messages,
            tools=tools,
            tool_choice="auto",
            store=True,
        )

        tool_calls = [item for item in response.output if item.type == "function_call"]
        # If the assistant returned standard content with no tool calls, we're done.
        if not tool_calls:
            return response, tool_metrics
            

        # Append the function call items from response output
        messages.extend(response.output)

        # Execute each resquested tool sequentially.
        for tool_call in tool_calls:
            fn_name = tool_call.name
            raw_args = tool_call.arguments or "{}"
            tool_start_ts = time.time()
            try:
                target_tool = tool_mapping.get(fn_name)
                parsed_args = json.loads(raw_args)
                tool_result = target_tool(**parsed_args)
                tool_result_str = json.dumps(tool_result)
            except Exception as e:
                tool_result_str = f"Error executing tool {fn_name}: {str(e)}\n"
                tool_result_str += traceback.format_exc()
            tool_duration = time.time() - tool_start_ts

            # Update tool metrics
            if fn_name not in tool_metrics:
                tool_metrics[fn_name] = {"count": 0, "durations": []}
            tool_metrics[fn_name]["count"] += 1
            tool_metrics[fn_name]["durations"].append(tool_duration)
            # Provide the tool output back to the model
            messages.append(
                _prepara_tool_result(
                    tool_call.call_id,
                    tool_result_str
                )
            )
    

## Helper Functions

In [24]:
def parse_evaluation_file(file_path: Path) -> list[dict[str, Any]]:
    """Parse XML evaluation file and return list of evaluation tasks"""
    try:
        # Parse trusted local XML evaluation file
        tree = ET.parse(file_path) # noqa: S314
        root = tree.getroot()
        evaluations = []

        # Check for task elements
        tasks = root.findall(".//task")
        for task in tasks:
            prompt_elem = task.find("prompt")
            response_elem = task.find("response")

            if prompt_elem is not None and response_elem is not None:
                eval_dict = {
                    "prompt": (prompt_elem.text or "").strip(),
                    "response": (response_elem.text or "").strip(),
                }
                evaluations.append(eval_dict)
        return evaluations
    except Exception as e:
        print(f"Error parsing evaluation file {file_path}: {e}")
        return []

In [25]:
evaluations = parse_evaluation_file("./evaluation.xml")

In [26]:
evaluations

[{'prompt': 'Calculate the compound interest on $10,000 invested at 5% annual interest rate, compounded monthly for 3 years. What is the final amount in dollars (rounded to 2 decimal places)?',
  'response': '11614.72'},
 {'prompt': 'A projectile is launched at a 45-degree angle with an initial velocity of 50 m/s. Calculate the total distance (in meters) it has traveled from the launch point after 2 seconds, assuming g=9.8 m/s². Round to 2 decimal places.',
  'response': '87.25'},
 {'prompt': 'A sphere has a volume of 500 cubic meters. Calculate its surface area in square meters. Round to 2 decimal places.',
  'response': '304.65'},
 {'prompt': 'Calculate the population standard deviation of this dataset: [12, 15, 18, 22, 25, 30, 35]. Round to 2 decimal places.',
  'response': '7.61'},
 {'prompt': 'Calculate the pH of a solution with a hydrogen ion concentration of 3.5 × 10^-5 M. Round to 2 decimal places.',
  'response': '4.46'},
 {'prompt': 'Calculate the monthly payment for a $200,0

In [27]:
def evaluate_single_task(
    task: dict[str, Any], tools: list[dict[str, Any]], task_index: int
) -> dict[str, Any]:
    """Evaluate a single task with the given tools."""
    start_time = time.time()

    # Run the task
    print(f"Task {task_index + 1}: Running task with prompt: {task['prompt']}")
    response, tool_metrics = agent_loop(task["prompt"], tools)

    # Extract all tagged content
    def _extract_xml_content(text, tag):
        pattern = rf"<{tag}>(.*?)</{tag}>"
        matches = re.findall(pattern, text, re.DOTALL)
        return matches[-1].strip() if matches else None

    response, summary, feedback = (
        _extract_xml_content(response.output_text, tag) for tag in ["response", "summary", "feedback"]
    )
    duration_seconds = time.time() - start_time

    return {
        "prompt": task["prompt"],
        "expected": task["response"],
        "actual": response,
        "score": int(response == task["response"]),
        "total_duration": duration_seconds,
        "tool_calls": tool_metrics,
        "num_tool_calls": sum(len(metrics["durations"]) for metrics in tool_metrics.values()),
        "summary": summary,
        "feedback": feedback,
    }

## Main Evaluation Function

In [28]:
# Report Templates
REPORT_HEADER = """
# Evaluation Report

## Summary

- **Accuracy**: {correct}/{total} ({accuracy:.1f}%)
- **Average Task Duration**: {average_duration_s:.2f}s
- **Average Tool Calls per Task**: {average_tool_calls:.2f}
- **Total Tool Calls**: {total_tool_calls}

---
"""

TASK_TEMPLATE = """
### Task

**Prompt**: {prompt}
**Ground Truth Response**: `{expected_response}`
**Actual Response**: `{actual_response}`
**Correct**: {correct_indicator}
**Duration**: {total_duration:.2f}s
**Tool Calls**: {tool_calls}

**Summary**
{summary}

**Feedback**
{feedback}

---
"""

def run_evaluation(eval_path: str, tools: list[dict[str, Any]]) -> str:
    """
    Run evaluation with provided tools using a simple loop.

    Args:
        eval_path: Path to XML evaluation file
        tools: List of tool definitions to use for evaluation
        
    """
    print("Starting Evaluation")

    eval_file = Path(eval_path)

    # Parse evaluation tasks
    tasks = parse_evaluation_file(eval_file)

    print(f"Loaded {len(tasks)} evaluation tasks")

    # Simple loop to run all tasks
    results = []
    for i, task in enumerate(tasks):
        print(f"Processing task {i + 1}/{len(tasks)}")
        results.append(evaluate_single_task(task, tools, i))

    # Calculate summary statistics
    correct = sum(r["score"] for r in results)
    accuracy = (correct / len(results)) * 100
    average_duration_s = sum(r["total_duration"] for r in results) / len(results)
    average_tool_calls = sum(r["num_tool_calls"] for r in results) / len(results)
    total_tool_calls = sum(r["num_tool_calls"] for r in results)

    report = REPORT_HEADER.format(
        correct=correct,
        total=len(results),
        accuracy=accuracy,
        average_duration_s=average_duration_s,
        average_tool_calls=average_tool_calls,
        total_tool_calls=total_tool_calls,
    )

    report += "".join(
        [
            TASK_TEMPLATE.format(
                prompt=task["prompt"],
                expected_response=task["response"],
                actual_response=result["actual"],
                correct_indicator="✅" if result["score"] else "❌",
                total_duration=result["total_duration"],
                tool_calls=json.dumps(result["tool_calls"], indent=2),
                summary=result["summary"] or "N/A",
                feedback=result["feedback"] or "N/A",
            )
            for task, result in zip(tasks, results, strict=False)
        ]
    )
    # Join all sections into final report
    return report

## Calculator Tool

In [29]:
def calculator(expression: str) -> str:
    """A basic calculator that performs arithmetic operations."""
    try:
        # Note: eval with restricted builtins is used here for demonstration.
        # In production, use a safer alernative like a math expression parser.
        result = eval(expression, {"__builtins__":{}}, {}) # noqa: S307
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

# Define the tool schema for the calculator
calculator_tool = {
    "type": "function",
    "name": "calculator",
    "description": "",
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "", # An unhelpful schema description
            }
        },
        "additionalProperties": False,
        "required": ["expression"], 
    },
}

# Set the tools list
tools = [calculator_tool]

In [30]:
tool_mapping = {
    "calculator": calculator
}

## Run Evaluation

In [31]:
# Run evaluation
print("Using calculator tool")

report = run_evaluation(eval_path="evaluation.xml", tools=tools)

print(report)

Using calculator tool
Starting Evaluation
Loaded 8 evaluation tasks
Processing task 1/8
Task 1: Running task with prompt: Calculate the compound interest on $10,000 invested at 5% annual interest rate, compounded monthly for 3 years. What is the final amount in dollars (rounded to 2 decimal places)?
Processing task 2/8
Task 2: Running task with prompt: A projectile is launched at a 45-degree angle with an initial velocity of 50 m/s. Calculate the total distance (in meters) it has traveled from the launch point after 2 seconds, assuming g=9.8 m/s². Round to 2 decimal places.
Processing task 3/8
Task 3: Running task with prompt: A sphere has a volume of 500 cubic meters. Calculate its surface area in square meters. Round to 2 decimal places.
Processing task 4/8
Task 4: Running task with prompt: Calculate the population standard deviation of this dataset: [12, 15, 18, 22, 25, 30, 35]. Round to 2 decimal places.
Processing task 5/8
Task 5: Running task with prompt: Calculate the pH of a so